## Import

In [ ]:
from utils import *
from sklearn.model_selection import train_test_split

## Parameters

In [ ]:
target_column = "AMZN"
input_days= 30
output_days= [1, 7, 14]
test_size=0.2
shuffle=True

## Load dataset

In [ ]:
data = read_and_clean_data(file_path = "./dataset/portfolio_data.csv", drop_list = ["DPZ", "BTC", "NFLX"])

In [ ]:
plot_data(data, "Amazon Stock Price", "Date", "AMZN")

### Generate Sliding Windows

In [ ]:
selected_value = data[target_column].values
window_data = {}
for item in output_days:
    X, y = generate_sliding_windows(
        selected_value,
        input_days,
        item
        )
    window_data[item] = {
        "X" : X,
        "y" : y
    }

In [ ]:
# window_output[1]["X"]

### Train / Test Split

In [ ]:
train_test_data = {}

for day in output_days:
    X = window_data[day]["X"]
    y = window_data[day]["y"]

    X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size= test_size,
            shuffle= shuffle
        )
    train_test_data[day] = {
            "X_train" : X_train,
            "X_test" : X_test,
            "y_train" : y_train,
            "y_test" : y_test
            }

In [ ]:
# print(train_test_data)
# print(train_test_data.keys)
# print(output_days)

### Data Scaling

In [ ]:
scaled_data = {}
print("Start")
for day in train_test_data:
    
    X_train_norm, X_test_norm, y_train_norm, y_test_norm, target_scaler = scale_data(
        train_test_data[day]["X_train"],
        train_test_data[day]["X_test"],
        train_test_data[day]["y_train"],
        train_test_data[day]["y_test"]
    )
    print(f"day- {day}")
    print(f"X_train Shape: {X_train_norm.shape} - X_test Shape: {X_test_norm.shape}")
    print(f"y_train Shape: {y_train_norm.shape} - y_test Shape: {y_test_norm.shape}")

    scaled_data[day] = {
        "X_train_norm" : X_train_norm,
        "X_test_norm" : X_test_norm,
        "y_train_norm" : y_train_norm,
        "y_test_norm" : y_test_norm,
        "target_scaler" : target_scaler
        }
    # print(scaled_data[day]["X_train_norm"])
    # print(scaled_data[day]["y_train_norm"])

### Training Regression Models

#### Dense Deep Network

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
dense_model = Sequential()
dense_model.add(Dense(32, input_shape=(30,), activation="relu"))
dense_model.add(Dense(64, activation="relu"))
dense_model.add(Dense(128, activation="relu"))
dense_model.add(Dense(64, activation="relu"))
dense_model.add(Dense(32, activation="relu"))
dense_model.add(Dense(10))



dense_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 32)             │           992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         2,112 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,104 (12.12 KB)

 Trainable params: 3,104 (12.12 KB)

 Non-trainable params: 0 (0.00 B)